In [2]:
import re
import json
import pandas as pd

class EmailExtractor:
    def extract(self, email):
     sender_match = re.search(r'From:\s*(.+?)(?:\n|$)', email, re.I)
     date_match = re.search(r'Date:\s*(\d{4}-\d{2}-\d{2})', email)
     amount_match = re.search(r'\$[\d,]+\.?\d*', email)
     subject_match = re.search(r'Subject:\s*(.+?)(?:\n|$)', email, re.I)

     return {
        'sender': sender_match.group(1) if sender_match else None,
        'date': date_match.group(1) if date_match else None,
        'amount': amount_match.group(0) if amount_match else None,
        'subject': subject_match.group(1) if subject_match else None,
    }

    def categorize(self, subject):
        if not subject:
            return 'Other'
        if any(w in subject.lower() for w in ['invoice', 'bill']):
            return 'Invoice'
        elif any(w in subject.lower() for w in ['receipt', 'order']):
            return 'Receipt'
        return 'Other'

    def process_batch(self, emails):
        results = [self.extract(e) for e in emails]
        for r in results:
            r['category'] = self.categorize(r.get('subject', ''))
            r['confidence'] = sum([1 for v in [r['sender'], r['date'], r['amount'], r['subject']] if v]) / 4
        return results

def generate_sample_emails(count=100):
    senders = ['amazon@amazon.com', 'stripe@stripe.com', 'github@github.com', 'aws@aws.com']
    subjects = ['Order Receipt', 'Invoice', 'Billing Confirmation', 'Payment']
    amounts = ['$50', '$100', '$250', '$500', '$1000']

    emails = []
    for i in range(count):
        email = f"""From: {senders[i % len(senders)]}
Date: 2025-01-{(i % 28) + 1:02d}
Subject: {subjects[i % len(subjects)]} #{i+1}

Amount: {amounts[i % len(amounts)]}"""
        emails.append(email)
    return emails

# RUN THIS
emails = generate_sample_emails(100)
extractor = EmailExtractor()
results = extractor.process_batch(emails)
df = pd.DataFrame(results)
df.to_csv('extracted.csv', index=False)

print(f"✅ Processed {len(results)} emails")
print(f"Success rate: {sum([r['confidence'] for r in results]) / len(results) * 100:.1f}%")


✅ Processed 100 emails
Success rate: 100.0%
